In [1]:
import os

PROJECT_DIR = (
    "/path/to/"
    "Maya Archarological Sites/Model Training/"
    "swin3d_project"
)

os.chdir(PROJECT_DIR)

print("Working directory:", os.getcwd())

Working directory: /path/to/Maya Archarological Sites/Model Training/swin3d_project


In [4]:
import torch

In [5]:
from pathlib import Path
from models.swin3d_model import Model
from datasets.belize_laz_dataset import create_dataloaders
from evaluate import evaluate_best_model

In [6]:
train_loader, validation_loader, test_loader = create_dataloaders(
    split_dir="dataset_splits",
    batch_size=8,
    voxel_max=65_000,
    num_workers=4,
)

print("Test batches:", len(test_loader))

Test batches: 10


In [7]:
train_loader, validation_loader, test_loader = create_dataloaders(
    split_dir="dataset_splits",
    batch_size=2,
    voxel_max=65_000,
    num_workers=4,
)

In [8]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = Model(
    num_classes=3,
    in_channels=4,
).to(device)

print("Device:", device)

Device: cuda


In [9]:
BEST_CHECKPOINT = Path("/path/to/Maya Archarological Sites/RepSurfMulticlass/segmentation/log/PointAnalysis/log/Slope3D/swin3d_25k_gdice_cosine_full/checkpoints/model_best.pth")

In [10]:
print("Checkpoint exists:", BEST_CHECKPOINT.exists())

Checkpoint exists: True


In [12]:
RESULTS_DIR = Path(
    "/path/to/project_storage/"
    "swin3d_results/repsurf"
)

In [17]:
BEST_CHECKPOINT = Path(
    "/path/to/project_storage/"
    "swin3d_checkpoints/"
    "generalized_dice/"
    "swin3d_generalized_dice_best_02.pth"
)

RESULTS_DIR = Path(
    "/path/to/project_storage/"
    "swin3d_results/generalized_dice"
)

print("Checkpoint exists:", BEST_CHECKPOINT.exists())
print("Checkpoint:", BEST_CHECKPOINT)
print("Results directory:", RESULTS_DIR)

Checkpoint exists: True
Checkpoint: /path/to/project_storage/swin3d_checkpoints/generalized_dice/swin3d_generalized_dice_best_02.pth
Results directory: /path/to/project_storage/swin3d_results/generalized_dice


In [13]:
(
    metrics,
    results_df,
    per_class_df,
    confusion_matrix_df,
    saved_paths,
) = evaluate_best_model(
    model=model,
    test_loader=test_loader,
    checkpoint_path=BEST_CHECKPOINT,
    output_dir=RESULTS_DIR,
    device=device,
    class_names=[
        "Ground",
        "Structure",
        "Plazuela",
    ],
    print_progress=True,
    save_results=True,
)

KeyError: "The checkpoint does not contain 'model_state_dict'."

In [19]:
display(results_df)

,Test Set: Overall (Pixel / Voxel) Accuracy (OA),Test Set: F1 Average,Test Set: F1 - Ground,Test Set: F1 - Structure,Test Set: F1 - Plazuelas,Test Set: Precision Average,Test Set: Precision - Ground,Test Set: Precision - Structures,Test Set: Precision - Plazuelas,Test Set: Recall Average,Test Set: Recall - Ground,Test Set: Recall - Structure,Test Set: Recall - Plazuelas,Test Set: mIoU (mean Intersection over Union),Test Set: IoU Ground,Test Set: IoU Structure,Test Set: IoU Plazuela,Test Set: Confusion Matrix TP / FP / FN / TN: Ground,Test Set: Confusion Matrix TP / FP / FN / TN: Structures,Test Set: Confusion Matrix TP / FP / FN / TN: Plazuelas
0,0.918349,0.525558,0.958748,0.136639,0.481288,0.508451,0.982352,0.176909,0.366094,0.583271,0.936252,0.111303,0.702259,0.437,0.920764,0.073329,0.316905,TP=4584833; FP=82369; FN=312177; TN=220621,TP=4180; FP=19448; FN=33375; TN=5142997,TP=186404; FP=322766; FN=79031; TN=4611799


In [20]:
display(per_class_df)

,Class,Precision,Recall,F1,IoU,TP,FP,FN,TN,Support
0,Ground,0.982352,0.936252,0.958748,0.920764,4584833,82369,312177,220621,4897010
1,Structure,0.176909,0.111303,0.136639,0.073329,4180,19448,33375,5142997,37555
2,Plazuela,0.366094,0.702259,0.481288,0.316905,186404,322766,79031,4611799,265435


In [21]:
display(confusion_matrix_df)

,Predicted Ground,Predicted Structure,Predicted Plazuela
Actual Ground,4584833,7550,304627
Actual Structure,15236,4180,18139
Actual Plazuela,67133,11898,186404


In [22]:
print(
    "Overall accuracy:",
    metrics["overall_accuracy"]
)

print(
    "Structure F1:",
    metrics["per_class"]["Structure"]["f1"]
)

print(
    "Plazuela IoU:",
    metrics["per_class"]["Plazuela"]["iou"]
)

Overall accuracy: 0.918349423076923
Structure F1: 0.1366392625402481
Plazuela IoU: 0.31690527557756615
